# Getting started with PetLongevity-Bench

This notebook loads the corpus and the candidate claim/evidence pairs, shows what the licence split means in practice, and checks that every shipped abstract matches its recorded SHA-256.

**Read first:** the pair annotation fields in v0.2 are placeholders filled from each paper's search category (every `label` is `supported`). Don't score models against them. What is reliable is the paper metadata, species, abstract text where licensed, offsets and hashes.

In [ ]:
import glob, hashlib, os
import pandas as pd

root = os.path.dirname(glob.glob('/kaggle/input/**/registry.csv', recursive=True)[0])
reg = pd.read_csv(f'{root}/registry.csv')
pairs = pd.read_csv(f'{root}/claim_evidence_pairs.csv')
print(len(reg), 'papers,', len(pairs), 'pairs')

## Corpus by species and search category

In [ ]:
reg.groupby(['species', 'target_category']).size().rename('papers').reset_index().sort_values('papers', ascending=False)

## Which text ships

Only CC BY abstracts are redistributed. Everything else carries the PMID, DOI and a hash so you can fetch it from PubMed yourself.

In [ ]:
reg['licence_family'].value_counts().rename('papers')

In [ ]:
pairs['text_withheld'].value_counts().rename('pairs')

## Verify the shipped abstracts

Character offsets refer to the exact abstract text, so a re-fetched abstract is only usable if its hash matches.

In [ ]:
shipped = reg[reg['text_included']]
ok = shipped.apply(lambda r: hashlib.sha256(r['abstract'].encode('utf-8')).hexdigest() == r['abstract_sha256'], axis=1)
print(f'{ok.sum()} / {len(shipped)} shipped abstracts match their hash')

## Recover a claim's supporting span from its offsets

In [ ]:
abstracts = shipped.set_index('paper_id')['abstract']
row = pairs[~pairs['text_withheld']].iloc[0]
span = abstracts[row['paper_id']][row['span_start_char']:row['span_end_char']]
print(row['pair_id'], '|', row['species'])
print(span)
assert span == row['claim_text']

## Next

Human-reviewed labels will replace the placeholders in a later version. A species-transfer probe built on these abstracts is being set up on Kaggle Benchmarks.